In [2]:
# 1. imports and paths

import os
import gzip
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

base       = 'D:/TNBC_SV_DNA_Repair'
data_dir   = os.path.join(base, 'dataset')
tables_dir = os.path.join(base, 'results', 'tables')
figures_dir= os.path.join(base, 'results', 'figures')
scores_dir = os.path.join(base, 'results', 'scores')

print('paths set')

paths set


In [4]:
# 2. load cohort

expr_tnbc  = pd.read_csv(os.path.join(tables_dir, 'expr_tnbc.csv'), index_col=0)
cn_tnbc    = pd.read_csv(os.path.join(tables_dir, 'cn_tnbc.csv'),   index_col=0)
surv_tnbc  = pd.read_csv(os.path.join(tables_dir, 'surv_tnbc.csv'))
mut_tnbc   = pd.read_csv(os.path.join(tables_dir, 'mut_tnbc.csv'))
cosmic_df  = pd.read_csv(os.path.join(tables_dir, 'panel_cosmic_annotation.csv'), index_col=0)

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

tnbc_samples = list(expr_tnbc.columns)

print('expression:', expr_tnbc.shape)
print('copy number:', cn_tnbc.shape)
print('survival:', surv_tnbc.shape)
print('mutations:', mut_tnbc.shape)
print('samples:', len(tnbc_samples))

expression: (26, 121)
copy number: (23, 121)
survival: (121, 11)
mutations: (14338, 12)
samples: 121


In [6]:
# 3. inspect gnomad header

gnomad_file = os.path.join(data_dir, 'gnomad.v4.1.sv.sites.bed.gz')

with gzip.open(gnomad_file, 'rt') as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 5:
            break

#chrom	start	end	name	svtype	samples	MULTIALLELIC	ALGORITHMS	BOTHSIDES_SUPPORT	CHR2	CPX_INTERVALS	CPX_TYPE	END	END2	EVIDENCE	LOW_CONFIDENCE_REPETITIVE_LARGE_DUP	MEMBERS	NCR	OUTLIER_SAMPLE_ENRICHED_LENIENT	PAR	PCRMINUS_NCR	PCRPLUS_NCR	PESR_GT_OVERDISPERSION	POS2	PREDICTED_BREAKEND_EXONIC	PREDICTED_COPY_GAIN	PREDICTED_DUP_PARTIAL	PREDICTED_INTERGENIC	PREDICTED_INTRAGENIC_EXON_DUP	PREDICTED_INTRONIC	PREDICTED_INV_SPAN	PREDICTED_LOF	PREDICTED_MSV_EXON_OVERLAP	PREDICTED_NEAREST_TSS	PREDICTED_NONCODING_BREAKPOINT	PREDICTED_NONCODING_SPAN	PREDICTED_PARTIAL_DISPERSED_DUP	PREDICTED_PARTIAL_EXON_DUP	PREDICTED_PROMOTER	PREDICTED_TSS_DUP	PREDICTED_UTR	RESOLVED_POSTHOC	SOURCE	SVLEN	SVTYPE	UNRESOLVED_TYPE	AN	AC	AF	N_BI_GENOS	N_HOMREF	N_HET	N_HOMALT	FREQ_HOMREF	FREQ_HET	FREQ_HOMALT	CN_NUMBER	CN_COUNT	CN_STATUS	CN_FREQ	CN_NONREF_COUNT	CN_NONREF_FREQ	AN_afr	AC_afr	AF_afr	N_BI_GENOS_afr	N_HOMREF_afr	N_HET_afr	N_HOMALT_afr	FREQ_HOMREF_afr	FREQ_HET_afr	FREQ_HOMALT_afr	CN_NUMBER_afr	CN_COUNT_afr	CN_STATUS_

In [10]:
# 4. load gnomad stream by gene overlap

gene_coords_temp = {
    'BRCA1':{'chr':'17','start':43044292,'end':43125483},
    'BRCA2':{'chr':'13','start':32315086,'end':32400268},
    'PALB2':{'chr':'16','start':23603160,'end':23641310},
    'RAD51':{'chr':'15','start':40696695,'end':40732498},
    'RAD51B':{'chr':'14','start':68096551,'end':68530339},
    'RAD51C':{'chr':'17','start':58670196,'end':58740640},
    'RAD51D':{'chr':'17','start':33445251,'end':33468065},
    'BRIP1':{'chr':'17','start':61679165,'end':61863518},
    'ATM':{'chr':'11','start':108222484,'end':108369102},
    'CHEK2':{'chr':'22','start':28687743,'end':28742422},
    'STAG2':{'chr':'X','start':123887078,'end':123985778},
    'STAG3':{'chr':'7','start':99578253,'end':99668848},
    'SMC1A':{'chr':'X','start':53380576,'end':53452974},
    'SMC1B':{'chr':'22','start':46459349,'end':46514419},
    'RAD21':{'chr':'8','start':117854195,'end':117900754},
    'REC8':{'chr':'14','start':23695940,'end':23749840},
    'HORMAD1':{'chr':'1','start':161613710,'end':161660197},
    'HORMAD2':{'chr':'22','start':30425078,'end':30491460},
    'SYCP2':{'chr':'20','start':8103374,'end':8228198},
    'SYCP3':{'chr':'12','start':52878906,'end':52901498},
    'MLH3':{'chr':'14','start':75516455,'end':75581972},
    'MSH4':{'chr':'1','start':93116500,'end':93214800},
    'MSH5':{'chr':'6','start':31924000,'end':31943000},
}

hits   = []
header = None

with gzip.open(gnomad_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            header = line.lstrip('#').rstrip().split('\t')
            continue
        parts = line.rstrip().split('\t')
        try:
            chrom = parts[0].replace('chr','')
            start = int(parts[1])
            end   = int(parts[2])
        except:
            continue
        for gene, coords in gene_coords_temp.items():
            if (chrom == str(coords['chr']) and
                start <= coords['end'] and
                end   >= coords['start']):
                row = dict(zip(header, parts)) if header and len(header)==len(parts) else {'raw': line.rstrip()}
                row['gene'] = gene
                hits.append(row)
                break

gnomad_hits = pd.DataFrame(hits)
print('gnomad hits at panel genes:', len(gnomad_hits))
if len(gnomad_hits) > 0:
    print('genes hit:', gnomad_hits['gene'].value_counts().to_dict())
    print('columns:', list(gnomad_hits.columns[:8]))

gnomad hits at panel genes: 2128
genes hit: {'RAD51B': 234, 'HORMAD1': 228, 'STAG3': 177, 'SYCP3': 136, 'STAG2': 127, 'BRIP1': 118, 'RAD51C': 107, 'BRCA1': 106, 'ATM': 96, 'MSH4': 95, 'SMC1A': 68, 'MSH5': 67, 'SMC1B': 64, 'SYCP2': 63, 'CHEK2': 60, 'REC8': 59, 'RAD21': 57, 'BRCA2': 56, 'HORMAD2': 55, 'RAD51': 46, 'PALB2': 46, 'MLH3': 45, 'RAD51D': 18}
columns: ['chrom', 'start', 'end', 'name', 'svtype', 'samples', 'MULTIALLELIC', 'ALGORITHMS']


In [13]:
# 6. gnomad bridge

# cell 4 already streamed hits directly
# gnomad_filt kept as alias for downstream cells
gnomad_filt = gnomad_hits.copy()

svtype_col = 'svtype'
chrom_col  = 'chrom'
start_col  = 'start'
end_col    = 'end'

gnomad_filt[start_col] = pd.to_numeric(gnomad_filt[start_col], errors='coerce')
gnomad_filt[end_col]   = pd.to_numeric(gnomad_filt[end_col],   errors='coerce')
gnomad_filt['sv_len']  = gnomad_filt[end_col] - gnomad_filt[start_col]
gnomad_filt['chrom_clean'] = gnomad_filt[chrom_col].astype(str).str.replace('chr','',regex=False)

print('gnomad_filt ready:', gnomad_filt.shape)
print('svtypes:', gnomad_filt[svtype_col].value_counts().head())

gnomad_filt ready: (2128, 629)
svtypes: svtype
DEL             934
CPX             370
DUP             225
BND             198
DEL:ME:LINE1    132
Name: count, dtype: int64


In [20]:
# 7. inspect HGSVC2 VCF

vcf_file = os.path.join(data_dir, 'variants_freeze4_sv_insdel_alt.vcf.gz')

with gzip.open(vcf_file, 'rt') as f:
    for i, line in enumerate(f):
        if line.startswith('##'):
            continue
        print(line.rstrip())
        if i > 20:
            break

#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	HG00096	HG00171	HG00512	HG00513	HG00514	HG00731	HG00732	HG00733	HG00864	HG01114	HG01505	HG01596	HG02011	HG02492	HG02587	HG02818	HG03009	HG03065	HG03125	HG03371	HG03486	HG03683	HG03732	NA12329	NA12878	NA18534	NA18939	NA19238	NA19239	NA19240	NA19650	NA19983	NA20509	NA20847	NA24385


In [25]:
# 8. load HGSVC2 stream by gene overlap

hgsvc_hits_list = []

with gzip.open(vcf_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.rstrip().split('\t')
        try:
            chrom = parts[0].replace('chr', '')
            start = int(parts[1])
            info  = parts[7]
        except:
            continue

        end    = start
        svtype = ''
        svlen  = 0

        for field in info.split(';'):
            if field.startswith('END='):
                try: end = int(field.split('=')[1])
                except: pass
            if field.startswith('SVTYPE='):
                svtype = field.split('=')[1]
            if field.startswith('SVLEN='):
                try: svlen = abs(int(field.split('=')[1]))
                except: pass

        # for insertions end == start, use start + svlen
        if end == start and svlen > 0:
            end = start + svlen

        if svlen < 50:
            continue

        for gene, coords in gene_coords_temp.items():
            if (chrom == str(coords['chr']) and
                start <= coords['end'] and
                end   >= coords['start']):
                hgsvc_hits_list.append({
                    'chrom':  chrom,
                    'start':  start,
                    'end':    end,
                    'sv_id':  parts[2],
                    'svtype': svtype,
                    'sv_len': svlen,
                    'filter': parts[6],
                    'gene':   gene
                })
                break

hgsvc_hits = pd.DataFrame(hgsvc_hits_list)
hgsvc_filt = hgsvc_hits.copy()

print('HGSVC2 hits at panel genes:', len(hgsvc_hits))
if len(hgsvc_hits) > 0:
    print('genes hit:', hgsvc_hits['gene'].value_counts().to_dict())
    print('svtypes:', hgsvc_hits['svtype'].value_counts().head())

HGSVC2 hits at panel genes: 49
genes hit: {'SMC1B': 8, 'RAD51B': 7, 'HORMAD1': 6, 'REC8': 4, 'RAD51C': 4, 'HORMAD2': 3, 'BRIP1': 3, 'STAG2': 2, 'SYCP2': 2, 'MSH5': 2, 'RAD21': 2, 'MSH4': 1, 'SYCP3': 1, 'RAD51D': 1, 'BRCA2': 1, 'CHEK2': 1, 'STAG3': 1}
svtypes: svtype
INS    27
DEL    22
Name: count, dtype: int64


In [22]:
# 8b. HGSVC2 diagnostic

count = 0
with gzip.open(vcf_file, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.rstrip().split('\t')
        try:
            chrom = parts[0]
            start = int(parts[1])
            info  = parts[7]
            end = start
            svtype = ''
            svlen = 0
            for field in info.split(';'):
                if field.startswith('END='):
                    try: end = int(field.split('=')[1])
                    except: pass
                if field.startswith('SVTYPE='):
                    svtype = field.split('=')[1]
                if field.startswith('SVLEN='):
                    try: svlen = abs(int(field.split('=')[1]))
                    except: pass
            print(f'chrom={chrom} start={start} end={end} svtype={svtype} svlen={svlen}')
        except:
            continue
        count += 1
        if count >= 10:
            break

chrom=chr1 start=10627 end=10627 svtype=INS svlen=58
chrom=chr1 start=66325 end=66325 svtype=INS svlen=160
chrom=chr1 start=66481 end=66481 svtype=INS svlen=82
chrom=chr1 start=90258 end=90258 svtype=INS svlen=59
chrom=chr1 start=121117 end=121117 svtype=INS svlen=113
chrom=chr1 start=126241 end=126241 svtype=DEL svlen=38630
chrom=chr1 start=136713 end=136713 svtype=DEL svlen=121
chrom=chr1 start=136789 end=136789 svtype=INS svlen=632
chrom=chr1 start=180283 end=180283 svtype=DEL svlen=224
chrom=chr1 start=180551 end=180551 svtype=DEL svlen=335


In [27]:
# 9. filter HGSVC2

# hgsvc_filt already created in cell 8
print('HGSVC2 filtered:', hgsvc_filt.shape)
print('svtypes:', hgsvc_filt['svtype'].value_counts().head())

HGSVC2 filtered: (49, 8)
svtypes: svtype
INS    27
DEL    22
Name: count, dtype: int64


In [29]:
# 10. gene loci coordinates

# GRCh38 coordinates for panel genes
gene_coords = {
    'BRCA1':   {'chr':'17', 'start':43044292,  'end':43125483},
    'BRCA2':   {'chr':'13', 'start':32315086,  'end':32400268},
    'PALB2':   {'chr':'16', 'start':23603160,  'end':23641310},
    'RAD51':   {'chr':'15', 'start':40696695,  'end':40732498},
    'RAD51B':  {'chr':'14', 'start':68096551,  'end':68530339},
    'RAD51C':  {'chr':'17', 'start':58670196,  'end':58740640},
    'RAD51D':  {'chr':'17', 'start':33445251,  'end':33468065},
    'BRIP1':   {'chr':'17', 'start':61679165,  'end':61863518},
    'ATM':     {'chr':'11', 'start':108222484, 'end':108369102},
    'CHEK2':   {'chr':'22', 'start':28687743,  'end':28742422},
    'STAG2':   {'chr':'X',  'start':123887078, 'end':123985778},
    'STAG3':   {'chr':'7',  'start':99578253,  'end':99668848},
    'SMC1A':   {'chr':'X',  'start':53380576,  'end':53452974},
    'SMC1B':   {'chr':'22', 'start':46459349,  'end':46514419},
    'RAD21':   {'chr':'8',  'start':117854195, 'end':117900754},
    'REC8':    {'chr':'14', 'start':23695940,  'end':23749840},
    'HORMAD1': {'chr':'1',  'start':161613710, 'end':161660197},
    'HORMAD2': {'chr':'22', 'start':30425078,  'end':30491460},
    'SYCP2':   {'chr':'20', 'start':8103374,   'end':8228198},
    'SYCP3':   {'chr':'12', 'start':52878906,  'end':52901498},
    'MLH3':    {'chr':'14', 'start':75516455,  'end':75581972},
    'MSH4':    {'chr':'1',  'start':93116500,  'end':93214800},
    'MSH5':    {'chr':'6',  'start':31924000,  'end':31943000},
}

print('gene coords defined:', len(gene_coords))

gene coords defined: 23


In [30]:
# 11. intersect gnomad with genes

chrom_col = list(gnomad_filt.columns)[0]
start_col = list(gnomad_filt.columns)[1]
end_col   = list(gnomad_filt.columns)[2]
svtype_col= next((c for c in gnomad_filt.columns if 'svtype' in c.lower()), gnomad_filt.columns[3])

gnomad_filt['chrom_clean'] = gnomad_filt[chrom_col].astype(str).str.replace('chr','',regex=False)

gnomad_gene_hits = []

for gene, coords in gene_coords.items():
    gc = str(coords['chr'])
    gs = coords['start']
    ge = coords['end']
    hits = gnomad_filt[
        (gnomad_filt['chrom_clean'] == gc) &
        (gnomad_filt[start_col] <= ge) &
        (gnomad_filt[end_col]   >= gs)
    ].copy()
    if len(hits) > 0:
        hits['gene'] = gene
        gnomad_gene_hits.append(hits)

if gnomad_gene_hits:
    gnomad_hits = pd.concat(gnomad_gene_hits, ignore_index=True)
else:
    gnomad_hits = pd.DataFrame()

print('gnomad hits at panel genes:', len(gnomad_hits))
if len(gnomad_hits) > 0:
    print('genes hit:', gnomad_hits['gene'].value_counts().head(10))

gnomad hits at panel genes: 2360
genes hit: gene
MSH4       272
RAD51B     234
HORMAD1    228
STAG3      177
SYCP3      136
BRIP1      129
STAG2      127
RAD51C     113
BRCA1      106
ATM         96
Name: count, dtype: int64


In [31]:
# 12. intersect HGSVC2 with genes

hgsvc_gene_hits = []

for gene, coords in gene_coords.items():
    gc = str(coords['chr'])
    gs = coords['start']
    ge = coords['end']
    hits = hgsvc_filt[
        (hgsvc_filt['chrom'] == gc) &
        (hgsvc_filt['start'] <= ge) &
        (hgsvc_filt['end']   >= gs)
    ].copy()
    if len(hits) > 0:
        hits['gene'] = gene
        hgsvc_gene_hits.append(hits)

if hgsvc_gene_hits:
    hgsvc_hits = pd.concat(hgsvc_gene_hits, ignore_index=True)
else:
    hgsvc_hits = pd.DataFrame()

print('HGSVC2 hits at panel genes:', len(hgsvc_hits))
if len(hgsvc_hits) > 0:
    print('genes hit:', hgsvc_hits['gene'].value_counts().head(10))

HGSVC2 hits at panel genes: 49
genes hit: gene
SMC1B      8
RAD51B     7
HORMAD1    6
REC8       4
RAD51C     4
BRIP1      3
HORMAD2    3
SYCP2      2
STAG2      2
RAD21      2
Name: count, dtype: int64


In [32]:
# 13. candidate locus set

gnomad_genes = set(gnomad_hits['gene'].unique()) if len(gnomad_hits) > 0 else set()
hgsvc_genes  = set(hgsvc_hits['gene'].unique())  if len(hgsvc_hits)  > 0 else set()

sv_candidate_genes = gnomad_genes | hgsvc_genes
sv_both            = gnomad_genes & hgsvc_genes
sv_longread_only   = hgsvc_genes - gnomad_genes

print('genes with gnomad SV evidence:', len(gnomad_genes))
print('genes with HGSVC2 SV evidence:', len(hgsvc_genes))
print('genes in both sources:', len(sv_both), '->', sorted(sv_both))
print('long-read only genes:', len(sv_longread_only), '->', sorted(sv_longread_only))
print('total candidate genes:', len(sv_candidate_genes))

# gene evidence table
ev_rows = []
for g in all_panel:
    ev_rows.append({
        'gene': g,
        'gnomad_sv': 'yes' if g in gnomad_genes else 'no',
        'hgsvc2_sv': 'yes' if g in hgsvc_genes  else 'no',
        'either_sv': 'yes' if g in sv_candidate_genes else 'no'
    })
sv_evidence = pd.DataFrame(ev_rows).set_index('gene')
print(sv_evidence)

genes with gnomad SV evidence: 23
genes with HGSVC2 SV evidence: 17
genes in both sources: 17 -> ['BRCA2', 'BRIP1', 'CHEK2', 'HORMAD1', 'HORMAD2', 'MSH4', 'MSH5', 'RAD21', 'RAD51B', 'RAD51C', 'RAD51D', 'REC8', 'SMC1B', 'STAG2', 'STAG3', 'SYCP2', 'SYCP3']
long-read only genes: 0 -> []
total candidate genes: 23
        gnomad_sv hgsvc2_sv either_sv
gene                                 
BRCA1         yes        no       yes
BRCA2         yes       yes       yes
PALB2         yes        no       yes
RAD51         yes        no       yes
RAD51B        yes       yes       yes
RAD51C        yes       yes       yes
RAD51D        yes       yes       yes
BRIP1         yes       yes       yes
ATM           yes        no       yes
CHEK2         yes       yes       yes
STAG2         yes       yes       yes
STAG3         yes       yes       yes
SMC1A         yes        no       yes
SMC1B         yes       yes       yes
RAD21         yes       yes       yes
REC8          yes       yes       yes
HORMA

In [33]:
# 14. expression z-score scoring

# compute per-gene z-scores across all TNBC samples
# outlier = |z| > 1.96 (5% tails)

expr_panel = expr_tnbc.loc[[g for g in all_panel if g in expr_tnbc.index]]

expr_z = expr_panel.apply(
    lambda row: stats.zscore(row, nan_policy='omit'), axis=1
)
expr_z = pd.DataFrame(
    expr_z.tolist(),
    index=expr_panel.index,
    columns=expr_panel.columns
)

# binary disruption: |z| > 1.96 means outlier expression
expr_disrupted = (expr_z.abs() > 1.96).astype(int)

print('expression z-score matrix:', expr_z.shape)
print('disrupted per gene (mean across samples):')
print(expr_disrupted.mean(axis=1).round(3))

expression z-score matrix: (23, 121)
disrupted per gene (mean across samples):
BRCA1      0.058
BRCA2      0.025
PALB2      0.041
RAD51      0.041
RAD51B     0.050
RAD51C     0.074
RAD51D     0.058
BRIP1      0.066
ATM        0.066
CHEK2      0.033
STAG2      0.041
STAG3      0.066
SMC1A      0.074
SMC1B      0.041
RAD21      0.041
REC8       0.050
HORMAD1    0.058
HORMAD2    0.058
SYCP2      0.074
SYCP3      0.050
MLH3       0.074
MSH4       0.050
MSH5       0.050
dtype: float64


In [34]:
# 15. copy number scoring

# GISTIC2 thresholded: -2 homdel, -1 hetloss, 0 neutral, 1 gain, 2 amp
# disrupted = not neutral

cn_panel = cn_tnbc.loc[[g for g in all_panel if g in cn_tnbc.index]]

cn_disrupted = (cn_panel != 0).astype(int)

# also note direction
cn_loss = (cn_panel < 0).astype(int)
cn_gain = (cn_panel > 0).astype(int)

print('cn disruption matrix:', cn_disrupted.shape)
print('fraction disrupted per gene:')
print(cn_disrupted.mean(axis=1).round(3))

cn disruption matrix: (23, 121)
fraction disrupted per gene:
Gene Symbol
BRCA1      0.711
BRCA2      0.545
PALB2      0.496
RAD51      0.463
RAD51B     0.562
RAD51C     0.636
RAD51D     0.694
BRIP1      0.653
ATM        0.636
CHEK2      0.537
STAG2      0.380
STAG3      0.479
SMC1A      0.388
SMC1B      0.595
RAD21      0.628
REC8       0.496
HORMAD1    0.711
HORMAD2    0.529
SYCP2      0.529
SYCP3      0.388
MLH3       0.537
MSH4       0.488
MSH5       0.438
dtype: float64


In [35]:
# 16. mutation scoring

# binary: 1 if sample has any nonsynonymous mutation in gene
nonsynon_effects = [
    'Missense_Mutation','Nonsense_Mutation','Frame_Shift_Del',
    'Frame_Shift_Ins','Splice_Site','In_Frame_Del','In_Frame_Ins',
    'Nonstop_Mutation','Translation_Start_Site',
    'nonsynonymous SNV','stopgain','stoploss','frameshift'
]

mut_damaging = mut_tnbc[
    mut_tnbc['effect'].isin(nonsynon_effects) |
    mut_tnbc['effect'].str.contains('Frame|Nonsense|Missense|Splice|Stop', na=False, case=False)
]

mut_disrupted = pd.DataFrame(
    0, index=all_panel, columns=tnbc_samples
)

for _, row in mut_damaging.iterrows():
    gene   = row['gene']
    sample = row['sample']
    if gene in mut_disrupted.index and sample in mut_disrupted.columns:
        mut_disrupted.loc[gene, sample] = 1

print('mutation disruption matrix:', mut_disrupted.shape)
print('fraction mutated per gene:')
print(mut_disrupted.mean(axis=1).round(3))

mutation disruption matrix: (23, 121)
fraction mutated per gene:
BRCA1      0.033
BRCA2      0.025
PALB2      0.025
RAD51      0.000
RAD51B     0.000
RAD51C     0.000
RAD51D     0.000
BRIP1      0.017
ATM        0.025
CHEK2      0.000
STAG2      0.017
STAG3      0.008
SMC1A      0.000
SMC1B      0.000
RAD21      0.000
REC8       0.017
HORMAD1    0.008
HORMAD2    0.000
SYCP2      0.017
SYCP3      0.000
MLH3       0.000
MSH4       0.017
MSH5       0.017
dtype: float64


In [36]:
# 17. align sample columns

common_samples = list(
    set(expr_disrupted.columns) &
    set(cn_disrupted.columns)   &
    set(mut_disrupted.columns)
)
common_samples.sort()

common_genes = list(
    set(expr_disrupted.index) &
    set(cn_disrupted.index)   &
    set(mut_disrupted.index)
)
common_genes = [g for g in all_panel if g in common_genes]

e = expr_disrupted.loc[common_genes, common_samples]
c = cn_disrupted.loc[common_genes,   common_samples]
m = mut_disrupted.loc[common_genes,  common_samples]

print('aligned samples:', len(common_samples))
print('aligned genes:', len(common_genes))

aligned samples: 121
aligned genes: 23


In [37]:
# 18. per-gene disruption score

# combine three evidence types with equal weights
# gene_score = (expr_disrupted + cn_disrupted + mut_disrupted) / 3
# range 0 to 1 per gene per sample

gene_score = (e + c + m) / 3.0

print('gene disruption score matrix:', gene_score.shape)
print('score range:', round(gene_score.values.min(),3), 'to', round(gene_score.values.max(),3))
print('mean score per gene:')
print(gene_score.mean(axis=1).round(3).sort_values(ascending=False))

gene disruption score matrix: (23, 121)
score range: 0.0 to 1.0
mean score per gene:
BRCA1      0.267
HORMAD1    0.259
RAD51D     0.251
BRIP1      0.245
ATM        0.242
RAD51C     0.237
RAD21      0.223
SMC1B      0.212
SYCP2      0.207
MLH3       0.204
RAD51B     0.204
BRCA2      0.198
HORMAD2    0.196
CHEK2      0.190
PALB2      0.187
REC8       0.187
STAG3      0.185
MSH4       0.185
RAD51      0.168
MSH5       0.168
SMC1A      0.154
STAG2      0.146
SYCP3      0.146
dtype: float64


In [38]:
# 19. compute GIPS

# GIPS = sum of gene disruption scores across all panel genes
# then scale 0 to 1 per patient

gips_raw = gene_score.sum(axis=0)

gips_min = gips_raw.min()
gips_max = gips_raw.max()
gips_scaled = (gips_raw - gips_min) / (gips_max - gips_min)

gips_df = pd.DataFrame({
    'sample':      common_samples,
    'GIPS_raw':    gips_raw.values,
    'GIPS_scaled': gips_scaled.values
})

print('GIPS computed:', gips_df.shape)
print('GIPS raw range:', round(gips_raw.min(),3), 'to', round(gips_raw.max(),3))
print('GIPS scaled mean:', round(gips_scaled.mean(),3))

GIPS computed: (121, 3)
GIPS raw range: 0.333 to 11.0
GIPS scaled mean: 0.406


In [39]:
# 20. stratify patients

# tertile-based stratification
t33 = gips_scaled.quantile(0.333)
t66 = gips_scaled.quantile(0.667)

def assign_group(x):
    if x <= t33:
        return 'Low'
    elif x <= t66:
        return 'Moderate'
    else:
        return 'High'

gips_df['GIPS_group'] = gips_scaled.apply(assign_group).values

print('GIPS stratification:')
print(gips_df['GIPS_group'].value_counts())
print('thresholds: low<=', round(t33,3), 'moderate<=', round(t66,3))

GIPS stratification:
GIPS_group
Low         52
High        36
Moderate    33
Name: count, dtype: int64
thresholds: low<= 0.344 moderate<= 0.5


In [40]:
# 21. add component scores

gips_df['expr_score'] = e.mean(axis=0).values
gips_df['cn_score']   = c.mean(axis=0).values
gips_df['mut_score']  = m.mean(axis=0).values

# add SV evidence flag per sample using gene-level evidence
sv_genes_list = list(sv_candidate_genes & set(common_genes))
if sv_genes_list:
    gips_df['sv_gene_disruption'] = gene_score.loc[sv_genes_list].mean(axis=0).values
else:
    gips_df['sv_gene_disruption'] = 0.0

print('gips_df columns:', list(gips_df.columns))
print(gips_df.head(3))

gips_df columns: ['sample', 'GIPS_raw', 'GIPS_scaled', 'GIPS_group', 'expr_score', 'cn_score', 'mut_score', 'sv_gene_disruption']
            sample  GIPS_raw  GIPS_scaled GIPS_group  expr_score  cn_score  \
0  TCGA-A1-A0SK-01  8.000000      0.71875       High    0.173913  0.869565   
1  TCGA-A1-A0SM-01  2.666667      0.21875        Low    0.000000  0.347826   
2  TCGA-A1-A0SO-01  9.000000      0.81250       High    0.130435  1.000000   

   mut_score  sv_gene_disruption  
0   0.000000            0.347826  
1   0.000000            0.115942  
2   0.043478            0.391304  


In [41]:
# 22. save results

gips_df.to_csv(os.path.join(scores_dir, 'gips_scores.csv'), index=False)
gene_score.to_csv(os.path.join(scores_dir, 'gene_disruption_scores.csv'))
sv_evidence.to_csv(os.path.join(tables_dir, 'sv_evidence_per_gene.csv'))

if len(gnomad_hits) > 0:
    gnomad_hits.to_csv(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv'), index=False)
if len(hgsvc_hits) > 0:
    hgsvc_hits.to_csv(os.path.join(tables_dir, 'hgsvc2_sv_gene_hits.csv'), index=False)

print('all results saved')

all results saved


In [42]:
# 23. GIPS distribution plot

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# histogram
axes[0].hist(gips_df['GIPS_scaled'], bins=25, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(t33, color='orange', linestyle='--', linewidth=1.2, label='tertile 1')
axes[0].axvline(t66, color='red',    linestyle='--', linewidth=1.2, label='tertile 2')
axes[0].set_xlabel('GIPS score')
axes[0].set_ylabel('patients')
axes[0].set_title('GIPS distribution')
axes[0].legend(fontsize=9)

# boxplot by group
group_order = ['Low', 'Moderate', 'High']
group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}
data_by_group = [gips_df[gips_df['GIPS_group']==g]['GIPS_scaled'].values for g in group_order]
bp = axes[1].boxplot(data_by_group, patch_artist=True, widths=0.5)
for patch, g in zip(bp['boxes'], group_order):
    patch.set_facecolor(group_colors[g])
    patch.set_alpha(0.7)
axes[1].set_xticklabels(group_order)
axes[1].set_ylabel('GIPS score')
axes[1].set_title('GIPS by group')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb2_gips_distribution.png'), dpi=150)
plt.show()
print('saved gips distribution')

saved gips distribution


In [43]:
# 24. gene disruption heatmap

# sort samples by GIPS
sample_order = gips_df.sort_values('GIPS_scaled')['sample'].tolist()
plot_data    = gene_score[sample_order]

# group color bar
group_map  = gips_df.set_index('sample')['GIPS_group']
group_col  = [group_colors[group_map[s]] for s in sample_order]

fig, ax = plt.subplots(figsize=(16, 8))

sns.heatmap(
    plot_data,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    yticklabels=True,
    xticklabels=False,
    linewidths=0,
    ax=ax,
    cbar_kws={'label': 'disruption score'}
)

ax.set_title('Gene disruption scores across TNBC cohort (sorted by GIPS)')
ax.set_xlabel('samples sorted by GIPS (low to high)')
ax.set_ylabel('panel gene')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb2_gene_disruption_heatmap.png'), dpi=150)
plt.show()
print('saved heatmap')

saved heatmap


In [44]:
# 25. component score barplot

comp_means = gips_df.groupby('GIPS_group')[['expr_score','cn_score','mut_score']].mean()
comp_means = comp_means.loc[group_order]

fig, ax = plt.subplots(figsize=(8, 5))

x     = np.arange(3)
width = 0.25
ax.bar(x - width, comp_means['expr_score'], width, label='expression', color='#5ba4cf')
ax.bar(x,         comp_means['cn_score'],   width, label='copy number', color='#f0a500')
ax.bar(x + width, comp_means['mut_score'],  width, label='mutation',   color='#d94f3d')

ax.set_xticks(x)
ax.set_xticklabels(group_order)
ax.set_ylabel('mean disruption score')
ax.set_title('Component scores by GIPS group')
ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb2_component_scores.png'), dpi=150)
plt.show()
print('saved component scores')

saved component scores


In [45]:
# 26. SV evidence summary plot

sv_plot = sv_evidence.copy()
sv_plot['group'] = [cosmic_df.loc[g,'group'] if g in cosmic_df.index else 'other' for g in sv_plot.index]
sv_plot['gnomad_n']  = [
    len(gnomad_hits[gnomad_hits['gene']==g]) if len(gnomad_hits)>0 and g in gnomad_hits['gene'].values else 0
    for g in sv_plot.index
]
sv_plot['hgsvc2_n']  = [
    len(hgsvc_hits[hgsvc_hits['gene']==g]) if len(hgsvc_hits)>0 and g in hgsvc_hits['gene'].values else 0
    for g in sv_plot.index
]

sv_plot_sorted = sv_plot.sort_values(['group','gnomad_n'], ascending=[True, False])

fig, ax = plt.subplots(figsize=(10, 6))
x     = np.arange(len(sv_plot_sorted))
width = 0.4
ax.bar(x - width/2, sv_plot_sorted['gnomad_n'], width, label='gnomAD SVs', color='#4878cf')
ax.bar(x + width/2, sv_plot_sorted['hgsvc2_n'], width, label='HGSVC2 SVs', color='#6acc65')
ax.set_xticks(x)
ax.set_xticklabels(sv_plot_sorted.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('SV count at locus')
ax.set_title('Population SV evidence at panel gene loci')
ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb2_sv_evidence_per_gene.png'), dpi=150)
plt.show()
print('saved SV evidence plot')

saved SV evidence plot


In [46]:
# 27. notebook 2 summary

summary2 = {
    'gnomad SVs at panel loci':    len(gnomad_hits) if len(gnomad_hits)>0 else 0,
    'hgsvc2 SVs at panel loci':    len(hgsvc_hits)  if len(hgsvc_hits)>0  else 0,
    'genes with SV evidence':       len(sv_candidate_genes),
    'long-read only genes':         len(sv_longread_only),
    'GIPS samples scored':          len(gips_df),
    'GIPS Low group':               int((gips_df['GIPS_group']=='Low').sum()),
    'GIPS Moderate group':          int((gips_df['GIPS_group']=='Moderate').sum()),
    'GIPS High group':              int((gips_df['GIPS_group']=='High').sum()),
}

for k, v in summary2.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(
    summary2, orient='index', columns=['value']
).to_csv(os.path.join(tables_dir, 'nb2_summary.csv'))

print('notebook 2 complete')

gnomad SVs at panel loci: 2360
hgsvc2 SVs at panel loci: 49
genes with SV evidence: 23
long-read only genes: 0
GIPS samples scored: 121
GIPS Low group: 52
GIPS Moderate group: 33
GIPS High group: 36
notebook 2 complete


In [5]:
import pandas as pd
import os

tables_dir = 'D:/TNBC_SV_DNA_Repair/results/tables'

gnomad_hits = pd.read_csv(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv'))
hgsvc2_hits = pd.read_csv(os.path.join(tables_dir, 'hgsvc2_sv_gene_hits.csv'))

print('gnomad columns:', gnomad_hits.columns.tolist())
print(gnomad_hits.head(3))
print()
print('hgsvc2 columns:', hgsvc2_hits.columns.tolist())
print(hgsvc2_hits.head(3))

gnomad columns: ['chrom', 'start', 'end', 'name', 'svtype', 'samples', 'MULTIALLELIC', 'ALGORITHMS', 'BOTHSIDES_SUPPORT', 'CHR2', 'CPX_INTERVALS', 'CPX_TYPE', 'END', 'END2', 'EVIDENCE', 'LOW_CONFIDENCE_REPETITIVE_LARGE_DUP', 'MEMBERS', 'NCR', 'OUTLIER_SAMPLE_ENRICHED_LENIENT', 'PAR', 'PCRMINUS_NCR', 'PCRPLUS_NCR', 'PESR_GT_OVERDISPERSION', 'POS2', 'PREDICTED_BREAKEND_EXONIC', 'PREDICTED_COPY_GAIN', 'PREDICTED_DUP_PARTIAL', 'PREDICTED_INTERGENIC', 'PREDICTED_INTRAGENIC_EXON_DUP', 'PREDICTED_INTRONIC', 'PREDICTED_INV_SPAN', 'PREDICTED_LOF', 'PREDICTED_MSV_EXON_OVERLAP', 'PREDICTED_NEAREST_TSS', 'PREDICTED_NONCODING_BREAKPOINT', 'PREDICTED_NONCODING_SPAN', 'PREDICTED_PARTIAL_DISPERSED_DUP', 'PREDICTED_PARTIAL_EXON_DUP', 'PREDICTED_PROMOTER', 'PREDICTED_TSS_DUP', 'PREDICTED_UTR', 'RESOLVED_POSTHOC', 'SOURCE', 'SVLEN', 'SVTYPE', 'UNRESOLVED_TYPE', 'AN', 'AC', 'AF', 'N_BI_GENOS', 'N_HOMREF', 'N_HET', 'N_HOMALT', 'FREQ_HOMREF', 'FREQ_HET', 'FREQ_HOMALT', 'CN_NUMBER', 'CN_COUNT', 'CN_STATUS', 

C:\Users\Koush\AppData\Local\Temp\ipykernel_13852\4117350548.py:6: DtypeWarning: Columns (26,32,37,42,57,58,59,73,74,75,89,90,91,105,106,107,121,122,123,137,138,139,153,154,155,169,170,171,185,186,187,201,202,203,217,218,219,239,240,241,259,260,261,275,276,277,295,296,297,311,312,313,331,332,333,347,348,349,367,368,369,383,384,385,403,404,405,419,420,421,439,440,441,455,456,457,475,476,477,491,492,493,511,512,513,527,528,529,547,548,549,563,564,565,583,584,585,599,600,601,619,620,621,628) have mixed types. Specify dtype option on import or set low_memory=False.
  gnomad_hits = pd.read_csv(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv'))


In [ ]:
# check other SV files 
import glob
sv_files = glob.glob(os.path.join(tables_dir, '*sv*'))
print('SV related files:', sv_files)

# also check scores dir
score_files = glob.glob(os.path.join(scores_dir, '*'))
print('Score files:', score_files)

SV related files: ['D:/TNBC_SV_DNA_Repair\\results\\tables\\cn_tnbc.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\cohort_summary.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\cox_multivariable_pfi.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\cox_os.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\cox_univariate_pfi.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\cross_cancer_spearman.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\enrichment_4mod_GO_Biological_Process_2023.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\enrichment_4mod_KEGG_2021_Human.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\enrichment_GO_Biological_Process_2023.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\enrichment_KEGG_2021_Human.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\expr_tnbc.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\gene_disruption_diff_highlow.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\gene_expression_panel.csv', 'D:/TNBC_SV_DNA_Repair\\results\\tables\\gnomad_sv_gene_hits.csv', 'D:/TNBC_

In [7]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from matplotlib.patches import Patch

base        = 'D:/TNBC_SV_DNA_Repair'
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

# Count SVs per gene from raw hit files
gnomad_hits = pd.read_csv(os.path.join(tables_dir, 'gnomad_sv_gene_hits.csv'),
                          low_memory=False)
hgsvc2_hits = pd.read_csv(os.path.join(tables_dir, 'hgsvc2_sv_gene_hits.csv'))

gnomad_counts = gnomad_hits.groupby('gene').size().rename('gnomad_n')
hgsvc2_counts = hgsvc2_hits.groupby('gene').size().rename('hgsvc2_n')

sv_counts_df = pd.DataFrame({'gnomad_n': gnomad_counts,
                              'hgsvc2_n': hgsvc2_counts}).fillna(0)
sv_counts_df = sv_counts_df.loc[sv_counts_df.index.isin(all_panel)]

print('SV counts per gene:')
print(sv_counts_df.sort_values('gnomad_n', ascending=False))

# Gene-level mean disruption
gene_score = pd.read_csv(os.path.join(scores_dir, 'gene_disruption_scores.csv'),
                         index_col=0)
gene_mean  = gene_score.mean(axis=1)

# Align
common     = sorted(set(sv_counts_df.index) & set(gene_mean.index))
sv_vals    = sv_counts_df.loc[common, 'gnomad_n'].values.astype(float)
dis_vals   = gene_mean[common].values.astype(float)

r, p = spearmanr(sv_vals, dis_vals)
print(f'\nSpearman r = {round(r,3)}, p = {round(p,4)}, n = {len(common)} genes')

# Scatter plot
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#d94f3d' if g in hr_genes else
          '#f0a500' if g in cohesin_genes else
          '#4878cf' for g in common]

ax.scatter(sv_vals, dis_vals, c=colors, s=70, alpha=0.85,
           edgecolors='white', linewidths=0.5)
for i, g in enumerate(common):
    ax.annotate(g, (sv_vals[i], dis_vals[i]),
                fontsize=7, alpha=0.75, xytext=(3,3),
                textcoords='offset points')

z = np.polyfit(sv_vals, dis_vals, 1)
x_line = np.linspace(sv_vals.min(), sv_vals.max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'k--', alpha=0.4, linewidth=1)

ax.set_xlabel('gnomAD SV count at gene locus')
ax.set_ylabel('Mean disruption score in TNBC patients')
ax.set_title(f'Population SV count vs TNBC disruption\nSpearman r={round(r,3)}, p={round(p,4)}')
ax.legend(handles=[Patch(facecolor='#d94f3d', label='HR'),
                   Patch(facecolor='#f0a500', label='Cohesin'),
                   Patch(facecolor='#4878cf', label='Meiosis')], fontsize=9)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb2_sv_disruption_correlation.png'), dpi=150)
plt.show()

pd.DataFrame({
    'gene':            common,
    'gnomad_sv_count': sv_vals,
    'mean_disruption': dis_vals,
    'group':           ['HR' if g in hr_genes else
                        'Cohesin' if g in cohesin_genes else
                        'Meiosis' for g in common]
}).to_csv(os.path.join(tables_dir, 'sv_disruption_correlation.csv'), index=False)
print('Saved sv_disruption_correlation.csv')

SV counts per gene:
         gnomad_n  hgsvc2_n
gene                       
MSH4          272       1.0
RAD51B        234       7.0
HORMAD1       228       6.0
STAG3         177       1.0
SYCP3         136       1.0
BRIP1         129       3.0
STAG2         127       2.0
RAD51C        113       4.0
BRCA1         106       0.0
ATM            96       0.0
SMC1A          90       0.0
MSH5           67       2.0
SMC1B          64       8.0
SYCP2          63       2.0
CHEK2          60       1.0
REC8           60       4.0
HORMAD2        57       3.0
RAD21          57       2.0
BRCA2          56       1.0
MLH3           53       0.0
PALB2          46       0.0
RAD51          46       0.0
RAD51D         23       1.0

Spearman r = -0.036, p = 0.8718, n = 23 genes
Saved sv_disruption_correlation.csv


C:\Users\Koush\AppData\Local\Temp\ipykernel_13852\1298861428.py:73: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
